# Making the Podcaster Crew Production-Ready

### Imports

In [ ]:
# !pip install -q "crewai[tools]" python-dotenv pydantic

import os
from datetime import datetime

from crewai import Agent, Crew, Process, Task
from crewai.llm import LLM
from crewai_tools import EXASearchTool
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = LLM(
    model="openai/gpt-4.1-2025-04-14",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

os.environ["EXA_API_KEY"] = os.getenv("EXA_API_KEY", "")
search_tool = EXASearchTool()
os.makedirs("task_outputs", exist_ok=True)

### 1. Structured Output

In [ ]:
researcher = Agent(
    role="AI Podcast Researcher",
    goal="Find recent releases, articles, topics, and progress about {topic}",
    backstory=(
        "You are a seasoned researcher with a knack for uncovering the latest developments. "
        "You find the most relevant information and present it clearly."
    ),
    llm=llm,
    tools=[search_tool],
    verbose=False,
)

In [ ]:
from pydantic import BaseModel, Field


class ResearchItem(BaseModel):
    title: str = Field(description="Short title for the finding")
    summary: str = Field(description="What happened or why it matters")
    listener_angle: str = Field(description="Why podcast listeners would care")
    source: str = Field(description="Source name or URL when available")


class ResearchReport(BaseModel):
    topic: str = Field(description="Podcast topic being researched")
    episode_angle: str = Field(description="Main angle for the episode")
    items: list[ResearchItem] = Field(description="Research findings for the episode")
    host_questions: list[str] = Field(
        description="Questions hosts can ask during the episode"
    )


structured_research_task = Task(
    description=(
        "Conduct podcast research about {topic}. Find recent releases, articles, topics, "
        "and progress. Use the current date: {date}."
    ),
    expected_output="A structured research report with an episode angle, findings, and host questions.",
    output_pydantic=ResearchReport,
    agent=researcher,
    output_file="task_outputs/structured_research.md",
)

structured_crew = Crew(
    agents=[researcher],
    tasks=[structured_research_task],
    process=Process.sequential,
    verbose=True,
)

inputs = {
    "topic": "AI Agents taking over job roles",
    "date": datetime.now().strftime("%B %d, %Y"),
}

structured_result = structured_crew.kickoff(inputs=inputs)
report = structured_result.pydantic
print(f"Episode angle: {report.episode_angle}")
print(f"Findings: {len(report.items)} | Host questions: {len(report.host_questions)}")

In [ ]:
report.topic

In [ ]:
report.episode_angle

In [ ]:
report.items

### 2. Guardrail

#### 2a. Code guardrail — validate the Pydantic model

In [ ]:
from typing import Any, Tuple

from crewai.tasks.task_output import TaskOutput


def validate_research_report(result: TaskOutput) -> Tuple[bool, Any]:
    report = result.pydantic
    if len(report.items) < 5:
        return (False, f"Need at least 5 findings, got {len(report.items)}. Search for more sources.")
    if not report.episode_angle:
        return (False, "Missing episode_angle. The report must include a clear angle for the episode.")
    for item in report.items:
        if not item.source:
            return (False, f"Finding '{item.title}' has no source URL. Every finding must cite a source.")
    return (True, report)


# output_pydantic gives us the structured model; guardrail validates it before accepting
research_task_validated = Task(
    description=(
        "Conduct podcast research with 5 items about {topic}. Find recent releases, articles, topics, "
        "and progress. Use the current date: {date}."
    ),
    expected_output="A structured research report with an episode angle, findings, and host questions.",
    output_pydantic=ResearchReport,
    guardrail=validate_research_report,
    agent=researcher,
    output_file="task_outputs/validated_research.md",
)

validated_crew = Crew(
    agents=[researcher],
    tasks=[research_task_validated],
    process=Process.sequential,
    verbose=True,
)

validated_result = validated_crew.kickoff(inputs={
    "topic": "AI Agents taking over job roles",
    "date": datetime.now().strftime("%B %d, %Y"),
})
validated_report = validated_result.pydantic
print(f"All checks passed — {len(validated_report.items)} findings, each with a source")


#### 2b. Plain-English guardrail — LLM-as-judge

In [ ]:
reporting_analyst = Agent(
    role="AI Podcast Reporting Analyst",
    goal="Create detailed reports based on {topic} research findings",
    backstory=(
        "You are a meticulous analyst with a keen eye for detail. "
        "You turn complex research into clear podcast briefing documents."
    ),
    llm=llm,
    verbose=False,
)

In [ ]:
# With guardrail — agent must retry until it meets the bar
# The guardrail is a plain-English string; CrewAI uses the LLM to judge the output
reporting_task_with_guardrail = Task(
    description=(
        "Review the research findings and write a detailed podcast briefing for {topic}."
    ),
    expected_output=(
        "A markdown report with at least 5 clearly labeled sections, "
        "each containing at least 3 bullet points."
    ),
    guardrail=(
        "Reject if the report has fewer than 5 sections with ## headers. "
        "Reject if any section has fewer than 3 bullet points. "
        "Each bullet point must contain a specific fact, not a vague statement."
    ),
    agent=reporting_analyst,
    context=[structured_research_task],
    output_file="task_outputs/podcast_briefing.md",
)

guardrail_crew = Crew(
    agents=[reporting_analyst],
    tasks=[reporting_task_with_guardrail],
    process=Process.sequential,
    verbose=True,
)

guardrail_result = guardrail_crew.kickoff(inputs={"topic": "AI Agents taking over job roles"})
print("Guardrail passed — briefing saved to task_outputs/podcast_briefing.md")

### 3. Add the Scriptwriter

In [ ]:
scriptwriter = Agent(
    role="Podcast Scriptwriter",
    goal="Create a fun, natural, interesting podcast script from a research report on {topic}",
    backstory=(
        "You are a naturally talented scriptwriter who turns technical reports into engaging, "
        "funny, and articulate podcast conversations between two hosts."
    ),
    llm=llm,
    verbose=False,
)

scripting_task = Task(
    description=(
        "Review the podcast briefing and write a script between two knowledgeable hosts {male_speaker} and {female_speaker}. "
        "First, identify the 3-4 most compelling moments. Then plan a narrative arc: cold open, "
        "topic order, and closing takeaway. Finally, write the full dialogue following that arc. "
        "Keep it lively, natural, and under 100 exchanges."
    ),
    expected_output=(
        "A complete podcast script with speaker labels for {male_speaker} and {female_speaker}, a cold open, "
        "topic transitions, and a closing summary."
    ),
    guardrail=(
        "The script must use exactly two distinct speaker labels consistently throughout — no mixing or switching names. "
        "It must include at least 12 lines of dialogue."
    ),
    agent=scriptwriter,
    context=[reporting_task_with_guardrail],
    output_file="task_outputs/podcast_script.md",
)

### 4. Full 3-Agent Pipeline

In [ ]:
research_task = Task(
    description=(
        "Conduct podcast research about {topic}. Find recent releases, articles, topics, "
        "and progress. Use the current date: {date}."
    ),
    expected_output="A structured research report with an episode angle, findings, and host questions.",
    output_pydantic=ResearchReport,
    agent=researcher,
    output_file="task_outputs/final_research.md",
)

final_reporting_task = Task(
    description="Review the research findings and write a detailed podcast briefing for {topic}.",
    expected_output=(
        "A markdown report with at least 5 clearly labeled sections, "
        "each containing at least 3 bullet points."
    ),
    guardrail=(
        "Reject if the report has fewer than 5 sections with ## headers. "
        "Reject if any section has fewer than 3 bullet points."
    ),
    agent=reporting_analyst,
    context=[research_task],
    output_file="task_outputs/final_briefing.md",
)

final_scripting_task = Task(
    description=(
        "Review the podcast briefing and write a script between two hosts {male_speaker} and {female_speaker}. "
        "Keep it lively, natural, and under 100 exchanges."
    ),
    expected_output="A complete podcast script with speaker labels for {male_speaker} and {female_speaker}.",
    guardrail=(
        "The script must use exactly two distinct speaker labels consistently throughout — no mixing or switching names. "
        "It must include at least 12 lines of dialogue."
    ),
    agent=scriptwriter,
    context=[final_reporting_task],
    output_file="task_outputs/final_script.md",
)

podcaster_crew = Crew(
    agents=[researcher, reporting_analyst, scriptwriter],
    tasks=[research_task, final_reporting_task, final_scripting_task],
    process=Process.sequential,
    verbose=True,
)

inputs = {
    "topic": "AI Agents taking over job roles",
    "date": datetime.now().strftime("%B %d, %Y"),
    "male_speaker": "Raj",
    "female_speaker": "Kiara"
}

_ = podcaster_crew.kickoff(inputs=inputs)
print("Pipeline complete — outputs saved to task_outputs/")